# Estimate LD mixture weights for GWAS

In [ ]:
Downloaded AFR + EUR meta-analyzed personality traits, we can use Susie to try to estimate the mixing proportion. By blocks. 

## Lift data

In [ ]:
Because the original data is in GRCH 37 we wanted to lift it to 38. So that matches with our reference. 

In [ ]:
/sc/arion/projects/ad-omics/haochen/rss_qc/real_data/meta_z_scores

ReGPC_agr_transancestry.txt.gz  ReGPC_con_transancestry.txt.gz  ReGPC_ext_transancestry.txt.gz  ReGPC_neu_transancestry.txt.gz  ReGPC_ope_transancestry.txt.gz

## Download chain file

In [ ]:
out_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/real_data/meta_z_scores_lifted"
tmp_dir="${out_dir}/tmp"
chain="${out_dir}/hg19ToHg38.over.chain.gz"
liftover_bin="${out_dir}/liftOver"

mkdir -p "${out_dir}" "${tmp_dir}"

wget -O "${liftover_bin}" https://hgdownload.soe.ucsc.edu/admin/exe/linux.x86_64.v479/liftOver
chmod +x "${liftover_bin}"
wget -O "${chain}" https://hgdownload.soe.ucsc.edu/goldenPath/hg19/liftOver/hg19ToHg38.over.chain.gz

In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/real_data/meta_z_scores_lifted"
mkdir -p ${work_dir}/{code,log,result}
cd ${work_dir}/code

cat << 'EOF' > base_script
#!/bin/bash
#BSUB -J "meta_lift[1-5]"
#BSUB -P acc_ad-omics
#BSUB -q express
#BSUB -n 1
#BSUB -W 2:00
#BSUB -R rusage[mem=8000]
#BSUB -o WORK_DIR/log/meta_lift.%J_%I.out
#BSUB -e WORK_DIR/log/meta_lift.%J_%I.err

set -euo pipefail

src_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/real_data/meta_z_scores"
out_dir="WORK_DIR/result"
tmp_dir="WORK_DIR/result/tmp_${LSB_JOBINDEX}"
mkdir -p ${out_dir} ${tmp_dir}

# ---- point these at a liftOver binary + hg19->hg38 chain on Minerva ----
liftOver_bin="WORK_DIR/liftOver"
chain_file="WORK_DIR/hg19ToHg38.over.chain.gz"

files=(
  ReGPC_agr_transancestry.txt.gz
  ReGPC_con_transancestry.txt.gz
  ReGPC_ext_transancestry.txt.gz
  ReGPC_neu_transancestry.txt.gz
  ReGPC_ope_transancestry.txt.gz
)
fname="${files[$((LSB_JOBINDEX-1))]}"
base="${fname%.txt.gz}"
in_file="${src_dir}/${fname}"
raw="${tmp_dir}/${base}.raw"

[ -f "${in_file}" ]      || { echo "Missing ${in_file}"; exit 1; }
[ -x "${liftOver_bin}" ] || { echo "liftOver not executable: ${liftOver_bin}"; exit 1; }
[ -f "${chain_file}" ]   || { echo "Missing chain: ${chain_file}"; exit 1; }

echo "Processing ${fname}"
zcat "${in_file}" > "${raw}"

# hg19 BED: chr, start(0-based), end, key=orig line number.  CHR=col2 POS=col3
awk 'function ucsc(c,  cc){ cc=c; sub(/^chr/,"",cc);
        if(cc=="23")cc="X"; else if(cc=="24")cc="Y"; else if(cc=="25")cc="X";
        else if(cc=="26"||cc=="MT"||cc=="M")cc="M"; return "chr" cc }
     BEGIN{OFS="\t"} NR>1 { print ucsc($2), $3-1, $3, NR }' \
     "${raw}" > "${tmp_dir}/${base}.hg19.bed"

"${liftOver_bin}" \
    "${tmp_dir}/${base}.hg19.bed" \
    "${chain_file}" \
    "${tmp_dir}/${base}.hg38.bed" \
    "${tmp_dir}/${base}.unmapped.bed"

# rebuild: update POS to hg38, keep same-chr mapped rows, drop unmapped/chr-changers
awk 'function ucsc(c,  cc){ cc=c; sub(/^chr/,"",cc);
        if(cc=="23")cc="X"; else if(cc=="24")cc="Y"; else if(cc=="25")cc="X";
        else if(cc=="26"||cc=="MT"||cc=="M")cc="M"; return "chr" cc }
     BEGIN{OFS="\t"}
     FNR==NR { nc[$4]=$1; np[$4]=$3; next }      # hg38 bed
     FNR==1  { $1=$1; print; next }              # header (normalize to tab)
     { k=FNR; if((k in np) && nc[k]==ucsc($2)) { $3=np[k]; print } }' \
     "${tmp_dir}/${base}.hg38.bed" "${raw}" \
     | gzip -c > "${out_dir}/${base}.txt.gz"

n_in=$(( $(wc -l < "${raw}") - 1 ))
n_out=$(( $(zcat "${out_dir}/${base}.txt.gz" | wc -l) - 1 ))
echo "${base}: lifted ${n_out} / ${n_in}" | tee "${out_dir}/${base}.summary.txt"
cp "${tmp_dir}/${base}.unmapped.bed" "${out_dir}/${base}.unmapped.bed"
rm -rf "${tmp_dir}"
echo "Done ${base}"
EOF

base_sh="base_script"
output_script="run_meta_liftover.sh"
sed "s|WORK_DIR|${work_dir}|g" ${base_sh} > ${output_script}
bsub < ${output_script}

## Partition sumstat into regions

In [ ]:
library(tidyverse)
region_path = "/sc/arion/projects/ad-omics/haochen/rss_qc/real_data/hg38_1362_blocks.bed"
blocks = read_delim(region_path)


sample_table = blocks %>%
  slice_sample(n = 500) %>% mutate(index = row_number())

sample_table %>% write_tsv("/sc/arion/projects/ad-omics/haochen/rss_qc/real_data/meta_z_scores_lifted/sampled_LD_block_region.tsv")

sumstat = read_delim("/sc/arion/projects/ad-omics/haochen/rss_qc/real_data/meta_z_scores_lifted/result/ReGPC_agr_transancestry.txt.gz")

#chr,start,end,block_id
<chr>,<dbl>,<dbl>,<chr>
chr1,101384274,104443097,chr1_101384274_104443097
chr1,104443097,106225286,chr1_104443097_106225286
chr1,106225286,109761915,chr1_106225286_109761915
chr1,109761915,111483530,chr1_109761915_111483530
chr1,111483530,113276642,chr1_111483530_113276642
chr1,113276642,115338054,chr1_113276642_115338054
chr1,11328222,12710318,chr1_11328222_12710318
chr1,115338054,117562321,chr1_115338054_117562321
chr1,117562321,119804207,chr1_117562321_119804207


In [ ]:
work_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/real_data/meta_z_scores_lifted"
mkdir -p ${work_dir}/{code,log}
mkdir -p ${work_dir}/regions_split
cd ${work_dir}/code

# ---- SET THIS: your region table (cols: chr start end block_id index; header row) ----
region_file="/sc/arion/projects/ad-omics/haochen/rss_qc/real_data/meta_z_scores_lifted/sampled_LD_block_region.tsv"

cat << 'EOF' > base_split
#!/bin/bash
#BSUB -J "meta_split[1-5]"
#BSUB -P acc_ad-omics
#BSUB -q express
#BSUB -n 1
#BSUB -W 5:00
#BSUB -R rusage[mem=15000]
#BSUB -o WORK_DIR/log/meta_split.%J_%I.out
#BSUB -e WORK_DIR/log/meta_split.%J_%I.err

set -euo pipefail

src_dir="WORK_DIR/result"                 # lifted hg38 sumstats
region_file="REGION_FILE"
out_root="WORK_DIR/regions_split"
tmp_dir="${out_root}/tmp_${LSB_JOBINDEX}"
mkdir -p ${tmp_dir}

files=(
  ReGPC_agr_transancestry.txt.gz
  ReGPC_con_transancestry.txt.gz
  ReGPC_ext_transancestry.txt.gz
  ReGPC_neu_transancestry.txt.gz
  ReGPC_ope_transancestry.txt.gz
)
fname="${files[$((LSB_JOBINDEX-1))]}"
base="${fname%.txt.gz}"
in_file="${src_dir}/${fname}"
trait_out="${out_root}/${base}"
mkdir -p ${trait_out}
raw="${tmp_dir}/${base}.raw"

[ -f "${in_file}" ]     || { echo "Missing ${in_file}"; exit 1; }
[ -f "${region_file}" ] || { echo "Missing region file ${region_file}"; exit 1; }

echo "Splitting ${fname}"
zcat "${in_file}" > "${raw}"
hdr=$(head -1 "${raw}")

# sort by chr,pos (keep header) so the split is sequential + within-region ordered
{ head -1 "${raw}"; tail -n +2 "${raw}" | sort -k2,2 -k3,3n; } > "${raw}.sorted"
mv "${raw}.sorted" "${raw}"

# pre-create each region file with the sumstat header
awk 'NR>1{print $5}' "${region_file}" | while read -r idx; do
    printf '%s\n' "${hdr}" > "${trait_out}/${base}.region${idx}.txt"
done

# assign each SNP to its region by position; half-open [start,end).  CHR=col2 POS=col3
awk -v outdir="${trait_out}" -v base="${base}" '
  BEGIN{OFS="\t"}
  FNR==NR {                                  # region table
    if(FNR==1) next
    rc=$1; sub(/^chr/,"",rc)
    n=++cnt[rc]; rs[rc,n]=$2+0; re[rc,n]=$3+0; ri[rc,n]=$5
    next
  }
  FNR==1 { next }                            # sumstat header (already written)
  {
    c=$2; sub(/^chr/,"",c); p=$3+0; m=cnt[c]
    for(i=1;i<=m;i++){
      if(p>=rs[c,i] && p<re[c,i]){
        idx=ri[c,i]
        if(idx!=prev){ if(prev!=""){close(of)} of=outdir"/"base".region"idx".txt"; prev=idx }
        print >> of; a++; break
      }
    }
  }
  END{ printf("assigned %d SNPs\n", a) > "/dev/stderr" }
' "${region_file}" "${raw}"

# drop header-only (empty) region files
find "${trait_out}" -name "${base}.region*.txt" | while read -r f; do
    [ "$(wc -l < "$f")" -le 1 ] && rm -f "$f"
done

n_in=$(( $(wc -l < "${raw}") - 1 ))
n_files=$(find "${trait_out}" -name "${base}.region*.txt" | wc -l)
echo "${base}: ${n_in} SNPs -> ${n_files} non-empty region files"
rm -rf "${tmp_dir}"
echo "Done ${base}"
EOF

sed -e "s|WORK_DIR|${work_dir}|g" -e "s|REGION_FILE|${region_file}|g" base_split > run_meta_split.sh
bsub < run_meta_split.sh

## Example code - fitting

In [ ]:

library(susieR)
library(data.table)
library(tidyverse)
library(BEDMatrix)
library(vroom)
library(mixsqp)
library(pecotmr)
library(pgenlibr)
library(matrixStats)
library(crayon)
library(Rfast)

# ---- local source overrides ----
source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")

# pecotmr GitHub version (conda not yet updated)
for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/",
                     full.names = TRUE, pattern = "\\.R$")) source(f)

# pecotmr GitHub version (conda not yet updated)
for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",
                     full.names = TRUE, pattern = "\\.R$")) source(f)
select <- dplyr::select

tb = read_delim("/hpc/users/sunh17/haochen/rss_qc/real_data/meta_z_scores_lifted/regions_split/ReGPC_agr_transancestry/ReGPC_agr_transancestry.region32.txt")


tb = tb  %>%
  distinct(POS, .keep_all = TRUE) %>% mutate(variant_id = paste0("chr", CHR, ":", POS, ":", A1, ":", A2)) %>%
  dplyr::rename(chrom = CHR, pos = POS) %>%
  dplyr::mutate(z = BETA / SE)



start = min(tb$pos)
end = max(tb$pos)
chrom = tb$chrom %>% unique()



ld_meta_file <- "/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/rss_ld_sketch/ld_meta_file.tsv"
region <- sprintf("chr%s:%d-%d", chrom, start, end)
EUR_sketch           <- load_LD_matrix(ld_meta_file, region, return_genotype = TRUE)
EUR_sketch$LD_matrix <- scale(EUR_sketch$LD_matrix, center = TRUE, scale = TRUE)

afr_ld_meta_file = "/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/ADSP_R5_AFR_ld_sketch/ld_meta_file.tsv"

region <- sprintf("chr%s:%d-%d", chrom, start, end)

AFR_sketch           <- load_LD_matrix(afr_ld_meta_file, region, return_genotype = TRUE)
AFR_sketch$LD_matrix <- scale(AFR_sketch$LD_matrix, center = TRUE, scale = TRUE)

sketch_harmonize <- function(target_data, ref_variants, sketch) {
  qc_data <- allele_qc(match_min_prop = 0, target_data = target_data, ref_variants = ref_variants)
  qced <- qc_data$target_data_qced
  idx  <- match(qced$variant_id, colnames(sketch))
  if (anyNA(idx)) stop("Some qced variant_id values not found in sketch columns.")
  sketch2 <- sketch[, idx, drop = FALSE]
  flip_index <- which(qced$variants_id_original != qced$variant_id)
  if (length(flip_index) > 0) sketch2[, flip_index] <- -sketch2[, flip_index, drop = FALSE]
  colnames(sketch2) <- qced$variants_id_original
  list(sumstat = qced, sketch = sketch2, flipped_prop = length(flip_index) / ncol(sketch2))
}

EUR_harm <- sketch_harmonize(target_data  = tb %>% select(-variant_id),
                         ref_variants = EUR_sketch$ref_panel %>% select(-variant_id),
                         sketch       = EUR_sketch$LD_matrix)

AFR_harm <- sketch_harmonize(target_data  = tb %>% select(-variant_id),
                         ref_variants = AFR_sketch$ref_panel %>% select(-variant_id),
                         sketch       = AFR_sketch$LD_matrix)

inter_var = intersect(EUR_harm$sumstat$variant_id, AFR_harm$sumstat$variant_id)

#-------------------------
# 1. tb
#-------------------------
tb_sub <- tb %>%
  slice(match(inter_var, variant_id))

#-------------------------
# 2. EUR sketch
#-------------------------
eur_idx <- match(inter_var, EUR_sketch$ref_panel$variant_id)

EUR_sketch_sub <- EUR_sketch
EUR_sketch_sub$ref_panel <- EUR_sketch$ref_panel[eur_idx, , drop = FALSE]
EUR_sketch_sub$LD_matrix <- EUR_sketch$LD_matrix[, eur_idx, drop = FALSE]

#-------------------------
# 3. AFR sketch
#-------------------------
afr_idx <- match(inter_var, AFR_sketch$ref_panel$variant_id)

AFR_sketch_sub <- AFR_sketch
AFR_sketch_sub$ref_panel <- AFR_sketch$ref_panel[afr_idx, , drop = FALSE]
AFR_sketch_sub$LD_matrix <- AFR_sketch$LD_matrix[, afr_idx, drop = FALSE]

res = susie_rss(z = tb_sub$z, X = list(EUR_sketch_sub$LD_matrix, AFR_sketch_sub$LD_matrix), n = median(tb_sub$N))
res$omega
z_max = max(abs(tb_sub$z))
EUR_LD = get_cormat(EUR_sketch_sub$LD_matrix)
AFR_LD = get_cormat(AFR_sketch_sub$LD_matrix)

    om <- res$omega
    R_manual <- om[1] * EUR_LD + om[2] * AFR_LD

susie_res = susie_rss(z = tb_sub$z, R = R_manual, n = median(tb_sub$N))

cs_number = length(susie_res$sets$cs)



## Run for all sumstat regions

In [ ]:
[Meta_Mix]
parameter: infile = paths
# One region-split sumstat (.txt) per job; each region processed independently.
parameter: cwd = path("output")
parameter: job_size = 10
parameter: walltime = "12h"
parameter: mem = "32G"
parameter: numThreads = 1
parameter: max_variants = 0     # 0 = use all variants (set e.g. 3000 to cap)
parameter: container = ""

# LD sketch meta files (EUR, AFR)
parameter: ld_meta_eur = path("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/rss_ld_sketch/ld_meta_file.tsv")
parameter: ld_meta_afr = path("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/ADSP_R5_AFR_ld_sketch/ld_meta_file.tsv")

input: infile, group_by = 1
output: f'{cwd:a}/{_input:bn}.result.tsv'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,
      mem = mem, cores = numThreads,
      tags = f'{step_name}_{_output[0]:bn}'

R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container

    # ---- Libraries ----
    library(tidyverse)
    library(data.table)
    library(vroom)
    library(mixsqp)
    library(matrixStats)
    library(crayon)
    library(Rfast)
    library(pgenlibr)
    library(GenomicRanges)
    library(SummarizedExperiment)
    library(Rsamtools)
    library(pecotmr)
    library(susieR)

    rename <- dplyr::rename
    select <- dplyr::select

    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/", full.names = TRUE, pattern = "\\.R$")) source(f)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",  full.names = TRUE, pattern = "\\.R$")) source(f)

    ld_meta_eur  <- "${ld_meta_eur}"
    ld_meta_afr  <- "${ld_meta_afr}"
    max_variants <- ${max_variants}

    # ---- Helper ----
    sketch_harmonize <- function(target_data, ref_variants, sketch) {
      qc_data <- allele_qc(match_min_prop = 0, target_data = target_data, ref_variants = ref_variants)
      qced <- qc_data$target_data_qced
      idx  <- match(qced$variant_id, colnames(sketch))
      if (anyNA(idx)) stop("Some qced variant_id values not found in sketch columns.")
      sketch2 <- sketch[, idx, drop = FALSE]
      flip_index <- which(qced$variants_id_original != qced$variant_id)
      if (length(flip_index) > 0) sketch2[, flip_index] <- -sketch2[, flip_index, drop = FALSE]
      colnames(sketch2) <- qced$variants_id_original
      list(sumstat = qced, sketch = sketch2, flipped_prop = length(flip_index) / ncol(sketch2))
    }

    # ---- Parse trait / region from filename ----
    in_file   <- ${_input:r}
    bn        <- sub("\\.txt$", "", basename(in_file))
    trait     <- sub("\\.region.*$", "", bn)
    region_id <- sub(".*\\.region", "", bn)

    analyze_region <- function() {
      tb <- read_delim(in_file, show_col_types = FALSE)
      if (max_variants > 0) tb <- head(tb, max_variants)

      tb <- tb %>%
        distinct(POS, .keep_all = TRUE) %>%
        mutate(variant_id = paste0("chr", CHR, ":", POS, ":", A1, ":", A2)) %>%
        rename(chrom = CHR, pos = POS) %>%
        mutate(z = BETA / SE) %>%
        filter(is.finite(z))

      chrom <- unique(tb$chrom); start <- min(tb$pos); end <- max(tb$pos)
      region <- sprintf("chr%s:%d-%d", chrom, start, end)

      EUR_sketch <- load_LD_matrix(ld_meta_eur, region, return_genotype = TRUE)
      EUR_sketch$LD_matrix <- scale(EUR_sketch$LD_matrix, center = TRUE, scale = TRUE)
      AFR_sketch <- load_LD_matrix(ld_meta_afr, region, return_genotype = TRUE)
      AFR_sketch$LD_matrix <- scale(AFR_sketch$LD_matrix, center = TRUE, scale = TRUE)

      EUR_harm <- sketch_harmonize(tb %>% select(-variant_id), EUR_sketch$ref_panel %>% select(-variant_id), EUR_sketch$LD_matrix)
      AFR_harm <- sketch_harmonize(tb %>% select(-variant_id), AFR_sketch$ref_panel %>% select(-variant_id), AFR_sketch$LD_matrix)

      inter_var <- intersect(EUR_harm$sumstat$variant_id, AFR_harm$sumstat$variant_id)
      if (length(inter_var) < 2) stop("fewer than 2 shared variants after harmonization")

      tb_sub <- tb %>% dplyr::slice(match(inter_var, variant_id))
      eur_idx <- match(inter_var, EUR_sketch$ref_panel$variant_id)
      afr_idx <- match(inter_var, AFR_sketch$ref_panel$variant_id)
      if (anyNA(eur_idx) || anyNA(afr_idx)) stop("intersected variant_id not found in a ref_panel")

      EUR_g <- EUR_sketch$LD_matrix[, eur_idx, drop = FALSE]
      AFR_g <- AFR_sketch$LD_matrix[, afr_idx, drop = FALSE]
      n_med <- median(tb_sub$N)

      # 1) mixture fit -> omega (EUR, AFR)
      res <- susie_rss(z = tb_sub$z, X = list(EUR_g, AFR_g), n = n_med)
      om  <- res$omega

      # 2) refit on omega-weighted LD -> credible sets
      R_manual  <- om[1] * get_cormat(EUR_g) + om[2] * get_cormat(AFR_g)
      susie_res <- susie_rss(z = tb_sub$z, R = R_manual, n = n_med)

      summary_tb <- tibble(
        trait = trait, region = region_id,
        chrom = as.character(chrom), start = start, end = end,
        n_variants = nrow(tb), n_intersect = length(inter_var), n_median = n_med,
        omega_EUR = om[1], omega_AFR = om[2],
        z_max = max(abs(tb_sub$z)),
        n_cs = length(susie_res$sets$cs),
        status = "ok", message = NA_character_
      )
      list(summary = summary_tb, res = res, susie_res = susie_res, inter_var = inter_var)
    }

    out <- tryCatch(analyze_region(), error = function(e) list(
      summary = tibble(
        trait = trait, region = region_id, chrom = NA_character_,
        start = NA_real_, end = NA_real_, n_variants = NA_integer_,
        n_intersect = NA_integer_, n_median = NA_real_,
        omega_EUR = NA_real_, omega_AFR = NA_real_, z_max = NA_real_,
        n_cs = NA_integer_, status = "error", message = conditionMessage(e)),
      res = NULL, susie_res = NULL, inter_var = NULL))

    out_tsv <- ${_output:r}
    dir.create(dirname(out_tsv), recursive = TRUE, showWarnings = FALSE)
    write_tsv(out$summary, out_tsv)

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output"
job_name="Personal_traits_meta_mix_proportion"
job_dir="${base_dir}/${job_name}"
notebook="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/23.Supp.Meta_Proportion_Real_GWAS.ipynb"
regions_root="/hpc/users/sunh17/haochen/rss_qc/real_data/meta_z_scores_lifted/regions_split"

max_variants=0   # 0 = all variants

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

# ===== Base LSF script template =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 4
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=60000]
#BSUB -W 72:00
#BSUB -o WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e WORK_DIR/log/JOB_NAME.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "NOTEBOOK" Meta_Mix \
    --infile $(echo REGIONS_ROOT/TRAIT/*.region*.txt) \
    --max_variants MAXVAR --mem 80G \
    --cwd WORK_DIR/result/TRAIT/ -j 4
EOF

chmod +x ${job_name}.sh
base_sh=${job_name}.sh
work_dir="${job_dir}"

# ===== One job per trait subfolder =====
for trait in ReGPC_agr_transancestry ReGPC_con_transancestry ReGPC_ext_transancestry ReGPC_neu_transancestry ReGPC_ope_transancestry; do
  output_script="${job_name}_${trait}.sh"
  sed -e "s|JOB_NAME|${job_name}_${trait}|g" \
      -e "s|WORK_DIR|${work_dir}|g" \
      -e "s|NOTEBOOK|${notebook}|g" \
      -e "s|REGIONS_ROOT|${regions_root}|g" \
      -e "s|TRAIT|${trait}|g" \
      -e "s|MAXVAR|${max_variants}|g" \
    "${base_sh}" > "${output_script}"
  bsub < "${output_script}"
done

In [ ]:
[Meta_Mix_top_var]
parameter: infile = paths
# One region-split sumstat (.txt) per job; each region processed independently.
parameter: cwd = path("output")
parameter: job_size = 10
parameter: walltime = "12h"
parameter: mem = "32G"
parameter: numThreads = 1
parameter: max_variants = 0     # 0 = use all variants (set e.g. 3000 to cap)
parameter: container = ""

# LD sketch meta files (EUR, AFR)
parameter: ld_meta_eur = path("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/rss_ld_sketch/ld_meta_file.tsv")
parameter: ld_meta_afr = path("/sc/arion/projects/ad-omics/haochen/rss_qc/genotypes/ADSP_R5_AFR_ld_sketch/ld_meta_file.tsv")

input: infile, group_by = 1
output: f'{cwd:a}/{_input:bn}.result.tsv'
task: trunk_workers = 1, trunk_size = job_size, walltime = walltime,
      mem = mem, cores = numThreads,
      tags = f'{step_name}_{_output[0]:bn}'

R: expand = '${ }', stdout = f"{_output:n}.stdout", stderr = f"{_output:n}.stderr", container = container

    # ---- Libraries ----
    library(tidyverse)
    library(data.table)
    library(vroom)
    library(mixsqp)
    library(matrixStats)
    library(crayon)
    library(Rfast)
    library(pgenlibr)
    library(GenomicRanges)
    library(SummarizedExperiment)
    library(Rsamtools)
    library(pecotmr)
    library(susieR)

    rename <- dplyr::rename
    select <- dplyr::select

    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/simulate_linreg.R")
    source("/sc/arion/projects/ad-omics/haochen/rss_qc/codes/code/misc.R")
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/pecotmr/R/", full.names = TRUE, pattern = "\\.R$")) source(f)
    for (f in list.files("/sc/arion/projects/ad-omics/haochen/rss_qc/susieR/R/",  full.names = TRUE, pattern = "\\.R$")) source(f)

    ld_meta_eur  <- "${ld_meta_eur}"
    ld_meta_afr  <- "${ld_meta_afr}"
    max_variants <- ${max_variants}

    # ---- Helper ----
    sketch_harmonize <- function(target_data, ref_variants, sketch) {
      qc_data <- allele_qc(match_min_prop = 0, target_data = target_data, ref_variants = ref_variants)
      qced <- qc_data$target_data_qced
      idx  <- match(qced$variant_id, colnames(sketch))
      if (anyNA(idx)) stop("Some qced variant_id values not found in sketch columns.")
      sketch2 <- sketch[, idx, drop = FALSE]
      flip_index <- which(qced$variants_id_original != qced$variant_id)
      if (length(flip_index) > 0) sketch2[, flip_index] <- -sketch2[, flip_index, drop = FALSE]
      colnames(sketch2) <- qced$variants_id_original
      list(sumstat = qced, sketch = sketch2, flipped_prop = length(flip_index) / ncol(sketch2))
    }

    # ---- Parse trait / region from filename ----
    in_file   <- ${_input:r}
    bn        <- sub("\\.txt$", "", basename(in_file))
    trait     <- sub("\\.region.*$", "", bn)
    region_id <- sub(".*\\.region", "", bn)

    analyze_region <- function() {
      tb <- read_delim(in_file, show_col_types = FALSE)

      tb <- tb %>%
        distinct(POS, .keep_all = TRUE) %>%
        mutate(variant_id = paste0("chr", CHR, ":", POS, ":", A1, ":", A2)) %>%
        rename(chrom = CHR, pos = POS) %>%
        mutate(z = BETA / SE) %>%
        filter(is.finite(z))

      # keep the top-|z| variants to bound runtime
      if (max_variants > 0 && nrow(tb) > max_variants) {
        tb <- tb %>% dplyr::slice_max(abs(z), n = max_variants, with_ties = FALSE)
      }

      chrom <- unique(tb$chrom); start <- min(tb$pos); end <- max(tb$pos)
      region <- sprintf("chr%s:%d-%d", chrom, start, end)

      EUR_sketch <- load_LD_matrix(ld_meta_eur, region, return_genotype = TRUE)
      EUR_sketch$LD_matrix <- scale(EUR_sketch$LD_matrix, center = TRUE, scale = TRUE)
      AFR_sketch <- load_LD_matrix(ld_meta_afr, region, return_genotype = TRUE)
      AFR_sketch$LD_matrix <- scale(AFR_sketch$LD_matrix, center = TRUE, scale = TRUE)

      EUR_harm <- sketch_harmonize(tb %>% select(-variant_id), EUR_sketch$ref_panel %>% select(-variant_id), EUR_sketch$LD_matrix)
      AFR_harm <- sketch_harmonize(tb %>% select(-variant_id), AFR_sketch$ref_panel %>% select(-variant_id), AFR_sketch$LD_matrix)

      inter_var <- intersect(EUR_harm$sumstat$variant_id, AFR_harm$sumstat$variant_id)
      if (length(inter_var) < 2) stop("fewer than 2 shared variants after harmonization")

      # subset tb, EUR, AFR to the SAME variants; drop any not matched in all three.
      tb_idx  <- match(inter_var, tb$variant_id)
      eur_idx <- match(inter_var, EUR_sketch$ref_panel$variant_id)
      afr_idx <- match(inter_var, AFR_sketch$ref_panel$variant_id)

      keep <- !is.na(tb_idx) & !is.na(eur_idx) & !is.na(afr_idx)
      if (sum(keep) < 2) stop("fewer than 2 variants matched across sumstat and both panels")
      tb_idx <- tb_idx[keep]; eur_idx <- eur_idx[keep]; afr_idx <- afr_idx[keep]

      tb_sub <- tb[tb_idx, , drop = FALSE]
      EUR_g  <- EUR_sketch$LD_matrix[, eur_idx, drop = FALSE]
      AFR_g  <- AFR_sketch$LD_matrix[, afr_idx, drop = FALSE]
      n_med  <- median(tb_sub$N)

      # 1) mixture fit -> omega (EUR, AFR)
      res <- susie_rss(z = tb_sub$z, X = list(EUR_g, AFR_g), n = n_med)
      om  <- res$omega

      # 2) refit on omega-weighted LD -> credible sets
      R_manual  <- om[1] * get_cormat(EUR_g) + om[2] * get_cormat(AFR_g)
      susie_res <- susie_rss(z = tb_sub$z, R = R_manual, n = n_med)

      summary_tb <- tibble(
        trait = trait, region = region_id,
        chrom = as.character(chrom), start = start, end = end,
        n_variants = nrow(tb), n_intersect = nrow(tb_sub), n_median = n_med,
        omega_EUR = om[1], omega_AFR = om[2],
        z_max = max(abs(tb_sub$z)),
        n_cs = length(susie_res$sets$cs),
        status = "ok", message = NA_character_
      )
      list(summary = summary_tb, res = res, susie_res = susie_res, inter_var = inter_var)
    }

    out <- tryCatch(analyze_region(), error = function(e) list(
      summary = tibble(
        trait = trait, region = region_id, chrom = NA_character_,
        start = NA_real_, end = NA_real_, n_variants = NA_integer_,
        n_intersect = NA_integer_, n_median = NA_real_,
        omega_EUR = NA_real_, omega_AFR = NA_real_, z_max = NA_real_,
        n_cs = NA_integer_, status = "error", message = conditionMessage(e)),
      res = NULL, susie_res = NULL, inter_var = NULL))

    out_tsv <- ${_output:r}
    dir.create(dirname(out_tsv), recursive = TRUE, showWarnings = FALSE)
    write_tsv(out$summary, out_tsv)

In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output"
job_name="Personal_traits_top_var_meta_mix_proportion"
job_dir="${base_dir}/${job_name}"
notebook="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/23.Supp.Meta_Proportion_Real_GWAS.ipynb"
regions_root="/hpc/users/sunh17/haochen/rss_qc/real_data/meta_z_scores_lifted/regions_split"

max_variants=1000   # 0 = all variants

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

# ===== Base LSF script template =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 4
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=60000]
#BSUB -W 72:00
#BSUB -o WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e WORK_DIR/log/JOB_NAME.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "NOTEBOOK" Meta_Mix_top_var \
    --infile $(echo REGIONS_ROOT/TRAIT/*.region*.txt) \
    --max_variants MAXVAR --mem 80G \
    --cwd WORK_DIR/result/TRAIT/ -j 4
EOF

chmod +x ${job_name}.sh
base_sh=${job_name}.sh
work_dir="${job_dir}"

# ===== One job per trait subfolder =====
for trait in ReGPC_agr_transancestry ReGPC_con_transancestry ReGPC_ext_transancestry ReGPC_neu_transancestry ReGPC_ope_transancestry; do
  output_script="${job_name}_${trait}.sh"
  sed -e "s|JOB_NAME|${job_name}_${trait}|g" \
      -e "s|WORK_DIR|${work_dir}|g" \
      -e "s|NOTEBOOK|${notebook}|g" \
      -e "s|REGIONS_ROOT|${regions_root}|g" \
      -e "s|TRAIT|${trait}|g" \
      -e "s|MAXVAR|${max_variants}|g" \
    "${base_sh}" > "${output_script}"
  bsub < "${output_script}"
done

## Restrict weight estimation to the 50 loci with the largest lead $|z|$

In [155]:
library(tidyverse)
library(data.table)

region_path <- "/sc/arion/projects/ad-omics/haochen/rss_qc/real_data/hg38_1362_blocks.bed"
sumstat_dir <- "/sc/arion/projects/ad-omics/haochen/rss_qc/real_data/meta_z_scores_lifted/result"
out_root    <- "/sc/arion/projects/ad-omics/haochen/rss_qc/real_data/meta_z_scores_lifted/regions_top50"
top_n       <- 50

traits <- c("ReGPC_agr_transancestry","ReGPC_con_transancestry","ReGPC_ext_transancestry",
            "ReGPC_neu_transancestry","ReGPC_ope_transancestry")

# ---- blocks: give each a stable global index; use half-open [start, end) ----
blocks <- read_delim(region_path, show_col_types = FALSE)
names(blocks)[1:4] <- c("chr","start","end","block_id")
blocks <- blocks %>%
  mutate(block_index = row_number(), chr = sub("^chr","", as.character(chr)))
setDT(blocks)
blocks[, `:=`(start_j = start, end_j = end - 1L)]      # end-1 so shared boundaries don't double-assign
setkey(blocks, chr, start_j, end_j)

partition_top <- function(trait) {
  in_file <- file.path(sumstat_dir, paste0(trait, ".txt.gz"))
  tb <- fread(in_file)                                  # SNP CHR POS A1 A2 BETA SE P N MAF EAF INFO
  orig_cols <- names(tb)                                # preserve exact input columns for output
  tb[, z := BETA / SE][is.finite(z)]
  tb[, `:=`(chr = as.character(CHR), pos1 = POS, pos2 = POS)]

  # assign each variant to its block
  ov <- foverlaps(tb, blocks,
                  by.x = c("chr","pos1","pos2"),
                  by.y = c("chr","start_j","end_j"),
                  type = "within", nomatch = NULL)

  # score blocks by lead |z|, keep top N
  blk <- ov[, .(max_abs_z = max(abs(z)), n_var = .N), by = .(block_index, block_id)]
  setorder(blk, -max_abs_z)
  top <- head(blk, top_n)

  out_dir <- file.path(out_root, trait)
  dir.create(out_dir, recursive = TRUE, showWarnings = FALSE)

  for (bi in top$block_index) {
    sub <- ov[block_index == bi]
    setorder(sub, POS)
    fwrite(sub[, ..orig_cols],
           file.path(out_dir, paste0(trait, ".region", bi, ".txt")), sep = "\t")
  }
  fwrite(top[, .(rank = .I, block_index, block_id, max_abs_z, n_var)],
         file.path(out_dir, paste0(trait, ".top", top_n, "_manifest.tsv")), sep = "\t")

  cat(sprintf("%s: %d blocks hit -> wrote top %d (max|z| %.2f .. %.2f)\n",
              trait, nrow(blk), nrow(top), max(top$max_abs_z), min(top$max_abs_z)))
}

walk(traits, partition_top)

ReGPC_agr_transancestry: 1361 blocks hit -> wrote top 50 (max|z| 9.68 .. 5.05)
ReGPC_con_transancestry: 1361 blocks hit -> wrote top 50 (max|z| 8.83 .. 5.76)
ReGPC_ext_transancestry: 1361 blocks hit -> wrote top 50 (max|z| 9.60 .. 6.20)
ReGPC_neu_transancestry: 1361 blocks hit -> wrote top 50 (max|z| 16.19 .. 8.06)
ReGPC_ope_transancestry: 1361 blocks hit -> wrote top 50 (max|z| 9.63 .. 5.82)


In [ ]:
# ===== Base config =====
base_dir="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output"
job_name="Personal_top50loci_top_var_meta_mix_proportion"
job_dir="${base_dir}/${job_name}"
notebook="/sc/arion/projects/ad-omics/haochen/rss_qc/2026_Simulation_Sinai/23.Supp.Meta_Proportion_Real_GWAS.ipynb"
regions_root="/sc/arion/projects/ad-omics/haochen/rss_qc/real_data/meta_z_scores_lifted/regions_top50"

max_variants=1000   # 0 = all variants

# ===== Create structure =====
mkdir -p "${job_dir}"/{code,result,log}
cd "${job_dir}/code"

# ===== Base LSF script template =====
cat << 'EOF' > ${job_name}.sh
#!/bin/bash -l
#BSUB -P acc_ad-omics
#BSUB -q premium
#BSUB -J JOB_NAME
#BSUB -n 4
#BSUB -R span[hosts=1]
#BSUB -R rusage[mem=60000]
#BSUB -W 72:00
#BSUB -o WORK_DIR/log/JOB_NAME.%J.out
#BSUB -e WORK_DIR/log/JOB_NAME.%J.err

export PATH=/sc/arion/projects/ad-omics/haochen/env/R_sc/bin:$PATH:/usr/bin:/bin

sos run "NOTEBOOK" Meta_Mix_top_var \
    --infile $(echo REGIONS_ROOT/TRAIT/*.region*.txt) \
    --max_variants MAXVAR --mem 80G \
    --cwd WORK_DIR/result/TRAIT/ -j 4
EOF

chmod +x ${job_name}.sh
base_sh=${job_name}.sh
work_dir="${job_dir}"

# ===== One job per trait subfolder =====
for trait in ReGPC_agr_transancestry ReGPC_con_transancestry ReGPC_ext_transancestry ReGPC_neu_transancestry ReGPC_ope_transancestry; do
  output_script="${job_name}_${trait}.sh"
  sed -e "s|JOB_NAME|${job_name}_${trait}|g" \
      -e "s|WORK_DIR|${work_dir}|g" \
      -e "s|NOTEBOOK|${notebook}|g" \
      -e "s|REGIONS_ROOT|${regions_root}|g" \
      -e "s|TRAIT|${trait}|g" \
      -e "s|MAXVAR|${max_variants}|g" \
    "${base_sh}" > "${output_script}"
  bsub < "${output_script}"
done

## Result

In [158]:
library(tidyverse)

results_dir <- "/sc/arion/projects/ad-omics/haochen/rss_qc/2026_output/Personal_top50loci_top_var_meta_mix_proportion/result"

files <- list.files(results_dir, "\\.result\\.tsv$", recursive = TRUE, full.names = TRUE)

# 1) combined: every region, every study (study taken from the subfolder name)
combined <- files %>%
  map_dfr(\(f) read_tsv(f, col_types = cols(.default = "c")) %>% mutate(.path = f)) %>%
  mutate(study = basename(dirname(.path))) %>%
  select(-.path) %>%
  relocate(study) %>%
  type_convert()




── Column specification ────────────────────────────────────────────────────────
cols(
  study = col_character(),
  trait = col_character(),
  region = col_double(),
  chrom = col_double(),
  start = col_double(),
  end = col_double(),
  n_variants = col_double(),
  n_intersect = col_double(),
  n_median = col_double(),
  omega_EUR = col_double(),
  omega_AFR = col_double(),
  z_max = col_double(),
  n_cs = col_double(),
  status = col_character(),
  message = col_character()
)



In [161]:
library(dplyr); library(ggplot2)

plot_df <- combined %>%
  filter(n_cs > 0) %>%
  group_by(trait) %>%
  summarise(mean_omega = mean(omega_AFR),
            se = sd(omega_AFR) / sqrt(n()),
            n  = n(), .groups = "drop") %>%
  arrange(desc(mean_omega)) %>%
  mutate(trait = factor(trait, levels = trait))
# saveRDS(plot_df, ".../Figure_data/omega_AFR_by_trait.rds")   # store plot-ready

X <- 7; Y <- 5
options(repr.plot.width = X, repr.plot.height = Y)
fig_name <- "omega_AFR_by_trait"

p <- ggplot(plot_df, aes(trait, mean_omega)) +
  geom_col(fill = "#4C72B0", width = 0.7) +
  geom_errorbar(aes(ymin = mean_omega - se, ymax = mean_omega + se), width = 0.2) +
  labs(x = NULL, y = expression(omega[AFR])) +
  theme_classic(base_size = 16) +
  theme(panel.grid  = element_blank(),
        axis.line   = element_line(colour = "grey20", linewidth = 0.4),
        axis.text.x = element_text(angle = 45, hjust = 1))
p

# ggsave(file.path(fig_dir, paste0(fig_name, ".png")), p, width = X, height = Y, dpi = 300)
# ggsave(file.path(fig_dir, paste0(fig_name, ".pdf")), p, width = X, height = Y, device = cairo_pdf)

plot without title